In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as shc
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering

In [11]:
df = pd.read_csv('world-data-2023.csv')

In [12]:
drop_features = ['Abbreviation', 'Capital/Major City', 'Latitude', 'Longitude',
                 'Official language','Density\n(P/Km2)','Calling Code','Currency-Code',
                 'CPI Change (%)','Birth Rate','Co2-Emissions','CPI Change (%)',
                 'Fertility Rate','Gross primary education enrollment (%)',
                 'Gross tertiary education enrollment (%)','Minimum wage',
                 'Out of pocket health expenditure','Physicians per thousand',
                 'Tax revenue (%)','Total tax rate','Maternal mortality ratio',
                 'Gasoline Price','Largest city','CPI']
df.drop(drop_features, axis=1, inplace=True)
df

,Country,Agricultural Land( %),Land Area(Km2),Armed Forces size,Forested Area (%),GDP,Infant mortality,Life expectancy,Population,Population: Labor force participation (%),Unemployment rate,Urban_population
0,Afghanistan,58.10%,"652,230","323,000",2.10%,"$19,101,353,833",47.9,64.5,"38,041,754",48.90%,11.12%,"9,797,273"
1,Albania,43.10%,"28,748","9,000",28.10%,"$15,278,077,447",7.8,78.5,"2,854,191",55.70%,12.33%,"1,747,593"
2,Algeria,17.40%,"2,381,741","317,000",0.80%,"$169,988,236,398",20.1,76.7,"43,053,054",41.20%,11.70%,"31,510,100"
3,Andorra,40.00%,468,NaN,34.00%,"$3,154,057,987",2.7,NaN,"77,142",NaN,NaN,"67,873"
4,Angola,47.50%,"1,246,700","117,000",46.30%,"$94,635,415,870",51.6,60.8,"31,825,295",77.50%,6.89%,"21,061,025"
...,...,...,...,...,...,...,...,...,...,...,...,...
190,Venezuela,24.50%,"912,050","343,000",52.70%,"$482,359,318,768",21.4,72.1,"28,515,829",59.70%,8.80%,"25,162,368"
191,Vietnam,39.30%,"331,210","522,000",48.10%,"$261,921,244,843",16.5,75.3,"96,462,106",77.40%,2.01%,"35,332,140"
192,Yemen,44.60%,"527,968","40,000",1.00%,"$26,914,402,224",42.9,66.1,"29,161,922",38.00%,12.91%,"10,869,523"
193,Zambia,32.10%,"752,618","16,000",65.20%,"$23,064,722,446",40.4,63.5,"17,861,030",74.60%,11.43%,"7,871,713"


In [13]:
df = df.dropna()
df['Agricultural Land( %)'] = df['Agricultural Land( %)'].str.replace('%', '').astype(float)
df['Land Area(Km2)'] = df['Land Area(Km2)'].str.replace(',', '').astype(int)
df['Armed Forces size'] = df['Armed Forces size'].str.replace(',','').astype(int)
df['Forested Area (%)'] = df['Forested Area (%)'].str.replace('%','').astype(float)
df['Population'] = df['Population'].str.replace(',','').astype(int)
df['Population: Labor force participation (%)'] = df['Population: Labor force participation (%)'].str.replace('%','').astype(float)
df['Unemployment rate'] = df['Unemployment rate'].str.replace('%','').astype(float)
df['Urban_population'] = df['Urban_population'].str.replace(',','').astype(int)
df['GDP'] = df['GDP'].str.replace('$', '')
df['GDP'] = df['GDP'].str.replace(',', '').astype('longlong')
df

C:\Users\compro\AppData\Local\Temp\ipykernel_1572\2311956460.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Agricultural Land( %)'] = df['Agricultural Land( %)'].str.replace('%', '').astype(float)
C:\Users\compro\AppData\Local\Temp\ipykernel_1572\2311956460.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Land Area(Km2)'] = df['Land Area(Km2)'].str.replace(',', '').astype(int)
C:\Users\compro\AppData\Local\Temp\ipykernel_1572\2311956460.py:4: SettingWithCopyWarning: 
A value is trying to be 

,Country,Agricultural Land( %),Land Area(Km2),Armed Forces size,Forested Area (%),GDP,Infant mortality,Life expectancy,Population,Population: Labor force participation (%),Unemployment rate,Urban_population
0,Afghanistan,58.1,652230,323000,2.1,19101353833,47.9,64.5,38041754,48.9,11.12,9797273
1,Albania,43.1,28748,9000,28.1,15278077447,7.8,78.5,2854191,55.7,12.33,1747593
2,Algeria,17.4,2381741,317000,0.8,169988236398,20.1,76.7,43053054,41.2,11.70,31510100
4,Angola,47.5,1246700,117000,46.3,94635415870,51.6,60.8,31825295,77.5,6.89,21061025
6,Argentina,54.3,2780400,105000,9.8,449663446954,8.8,76.5,44938712,61.3,9.79,41339571
...,...,...,...,...,...,...,...,...,...,...,...,...
190,Venezuela,24.5,912050,343000,52.7,482359318768,21.4,72.1,28515829,59.7,8.80,25162368
191,Vietnam,39.3,331210,522000,48.1,261921244843,16.5,75.3,96462106,77.4,2.01,35332140
192,Yemen,44.6,527968,40000,1.0,26914402224,42.9,66.1,29161922,38.0,12.91,10869523
193,Zambia,32.1,752618,16000,65.2,23064722446,40.4,63.5,17861030,74.6,11.43,7871713


In [17]:
features = df.drop(columns=['Country'])
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

In [18]:
scaled_features

array([[ 0.82009655, -0.06848749,  0.4235004 , ..., -1.33688577,
         0.87981237, -0.19369196],
       [ 0.13110061, -0.37411599, -0.39744889, ..., -0.6788117 ,
         1.12589545, -0.2948816 ],
       [-1.04937908,  0.7793122 ,  0.40781347, ..., -2.08205788,
         0.99776955,  0.07925221],
       ...,
       [ 0.20000021, -0.12940025, -0.31639976, ..., -2.3917398 ,
         1.24385262, -0.18021309],
       [-0.37416307, -0.01927768, -0.37914747, ...,  1.15024712,
         0.94285845, -0.21789748],
       [ 0.07598094, -0.19666056, -0.28764039, ...,  1.97283971,
        -0.37500795, -0.25755041]])